In [11]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
# UserProxyAgent는 AI가 research할 때 대화하면서 진행할 수 있게 해줌
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from tools import web_search_tool, save_report_to_md

In [12]:
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [13]:
research_planner = AssistantAgent(
    "research_planner",
    description="A strategic research coordinator that breaks down complex questions into research subtasks",
    model_client=model_client,
    system_message="""You are a research planning specialist. Your job is to create a focused research plan.

For each research question, create a FOCUSED research plan with:

1. **Core Topics**: 2-3 main areas to investigate
2. **Search Queries**: Create 3-5 specific search queries covering:
   - Latest developments and news
   - Key statistics or data
   - Expert analysis or studies
   - Future outlook

Keep the plan focused and achievable. Quality over quantity.""",
)

research_agent = AssistantAgent(
    "research_agent",
    description="A web research specialist that searches and extracts content",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""You are a web research specialist. Your job is to conduct focused searches based on the research plan.

RESEARCH STRATEGY:
1. **Execute 3-5 searches** from the research plan
2. **Extract key information** from the results:
   - Main facts and statistics
   - Recent developments
   - Expert opinions
   - Important context

3. **Quality focus**:
   - Prioritize authoritative sources
   - Look for recent information (within 2 years)
   - Note diverse perspectives

After completing the searches from the plan, summarize what you found. Your goal is to gather 5-10 quality sources.""",
)

research_analyst = AssistantAgent(
    "research_analyst",
    description="An expert analyst that creates research reports",
    model_client=model_client,
    system_message="""You are a research analyst. Create a comprehensive report from the gathered research.

CREATE A RESEARCH REPORT with:

## Executive Summary
- Key findings and conclusions
- Main insights

## Background & Current State
- Current landscape
- Recent developments
- Key statistics and data

## Analysis & Insights
- Main trends
- Different perspectives
- Expert opinions

## Future Outlook
- Emerging trends
- Predictions
- Implications

## Sources
- List all sources used

Write a clear, well-structured report based on the research gathered. End with "REPORT_COMPLETE" when finished.""",
)

quality_reviewer = AssistantAgent(
    "quality_reviewer",
    description="A quality assurance specialist that evaluates research completeness and accuracy",
    tools=[save_report_to_md],
    model_client=model_client,
    system_message="""You are a quality reviewer. Your job is to check if the research analyst has produced a complete research report.

Look for:
- A comprehensive research report from the research analyst that ends with "REPORT_COMPLETE"
- The research question is fully answered
- Sources are cited and reliable
- The report includes summary, key information, analysis, and sources

When you see a complete research report that ends with "REPORT_COMPLETE":
1. First, use the save_report_to_md tool to save the report to report.md
2. Then say: "The research is complete. The report has been saved to report.md. Please review the report and let me know if you approve it or need additional research."

If the research analyst has NOT yet created a complete report, tell them to create one now.""",
)

research_enhancer = AssistantAgent(
    "research_enhancer",
    description="A specialist that identifies critical gaps only",
    model_client=model_client,
    system_message="""You are a research enhancement specialist. Your job is to identify ONLY CRITICAL gaps.

Review the research and ONLY suggest additional searches if there are MAJOR gaps like:
- Completely missing recent developments (last 6 months)
- No statistics or data at all
- Missing a crucial perspective that was specifically asked for

If the research covers the basics reasonably well, say: "The research is sufficient to proceed with the report."

Only suggest 1-2 additional searches if absolutely necessary. We prioritize getting a good report done rather than perfect coverage.""",
)

user_proxy = UserProxyAgent(
    "user_proxy",
    description="Human reviewer who can request additional research or approve final results",
    input_func=input,
)

In [14]:
selector_prompt = """
Choose the best agent for the current task based on the conversation history:

{roles}

Current conversation:
{history}

Available agents:
- research_planner: Plan the research approach (ONLY at the start)
- research_agent: Search for and extract content from web sources (after planning)
- research_enhancer: Identify CRITICAL gaps only (use sparingly)
- research_analyst: Write the final research report
- quality_reviewer: Check if a complete report exists
- user_proxy: Ask the human for feedback

WORKFLOW:
1. If no planning done yet → select research_planner
2. If planning done but no research → select research_agent  
3. After research_agent completes initial searches → select research_enhancer ONCE
4. If enhancer says "sufficient to proceed" → select research_analyst
5. If enhancer suggests critical searches → select research_agent ONCE more then research_analyst
6. If research_analyst said "REPORT_COMPLETE" → select quality_reviewer
7. If quality_reviewer asked for user feedback → select user_proxy

IMPORTANT: After research_agent has searched 2 times maximum, proceed to research_analyst regardless.

Pick the agent that should work next based on this workflow."""

In [15]:
text_termination = TextMentionTermination("APPROVED")
max_message_termination = MaxMessageTermination(max_messages=50)
termination_condition = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_agent,
        research_analyst,
        research_enhancer,
        research_planner,
        quality_reviewer,
        user_proxy,
    ],
    selector_prompt=selector_prompt,
    model_client=model_client,
    # allow_repeated_speaker=True
    termination_condition=termination_condition
)

In [16]:
await Console(team.run_stream(task="Research about the new development in Nuclear Energy"))

---------- TextMessage (user) ----------
Research about the new development in Nuclear Energy
---------- TextMessage (research_planner) ----------
### Research Plan: New Developments in Nuclear Energy

#### Core Topics:
1. **Advancements in Nuclear Reactor Technology**
   - Focus on small modular reactors (SMRs) and next-generation reactors.
  
2. **Safety and Environmental Impact**
   - Investigate recent improvements in safety protocols and environmental assessments related to nuclear energy.

3. **Global Policy and Market Dynamics**
   - Examine the policies supporting nuclear energy development in various countries and the role of the private sector.

#### Search Queries:
1. **Latest developments and news**:
   - "2023 advancements in nuclear reactor technology small modular reactors"
   - "New nuclear energy projects announced in 2023 worldwide"

2. **Key statistics or data**:
   - "2023 nuclear energy production statistics global report"
   - "Current nuclear energy investment tr

TaskResult(messages=[TextMessage(id='2627c1a9-8506-4137-8ada-7c801ca2ce58', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 2, 1, 6, 14, 45, 391516, tzinfo=datetime.timezone.utc), content='Research about the new development in Nuclear Energy', type='TextMessage'), TextMessage(id='82654ec1-b6ae-4aff-82fb-1be85266fdb0', source='research_planner', models_usage=RequestUsage(prompt_tokens=121, completion_tokens=286), metadata={}, created_at=datetime.datetime(2026, 2, 1, 6, 14, 53, 258365, tzinfo=datetime.timezone.utc), content='### Research Plan: New Developments in Nuclear Energy\n\n#### Core Topics:\n1. **Advancements in Nuclear Reactor Technology**\n   - Focus on small modular reactors (SMRs) and next-generation reactors.\n  \n2. **Safety and Environmental Impact**\n   - Investigate recent improvements in safety protocols and environmental assessments related to nuclear energy.\n\n3. **Global Policy and Market Dynamics**\n   - Examine the policies suppor